# Prices and Returns

ราคาหุ้นลดลงหลังจ่ายปันผล ผู้ลงทุนขาดทุนเท่ากับราคาที่ลดลงหรือไม่?

> “ราคาสินทรัพย์เปลี่ยนแปลงอยู่เสมอเมื่อตลาดการเงินเปิดทำการ”
>
> <span lang="en">“Asset prices are dynamic, changing frequently whenever the financial markets are open.”</span>
>
> — **Stephen J. Taylor** · [*Asset Price Dynamics, Volatility, and Prediction*, Preface, น. xiii](https://www.lancaster.ac.uk/people/afasjt/apdvp_contents.pdf#page=8) · แปลไทยเพื่อประกอบบทเรียน

In [1]:
"""Standard-library numerical examples; all observations are simulated."""
import math
import statistics


def uniform(seed):
    state = seed & 0xffffffff
    while True:
        state = (1664525*state + 1013904223) & 0xffffffff
        yield (state+.5)/4294967296


def normal_generator(seed):
    u = uniform(seed)
    while True:
        yield math.sqrt(-2*math.log(next(u)))*math.cos(2*math.pi*next(u))


def return_pair(previous, current, dividend=0):
    assert previous > 0 and current+dividend > 0
    simple = (current+dividend)/previous-1
    return simple, math.log1p(simple)


def moments(values):
    average = statistics.mean(values)
    centered = [x-average for x in values]
    m2 = statistics.mean(x*x for x in centered)
    m4 = statistics.mean(x**4 for x in centered)
    return {'mean': average, 'sd': statistics.stdev(values), 'variance': m2,
            'kurtosis': m4/m2**2 if m2 > 0 else None}


def acf(values, max_lag=20):
    assert 0 <= max_lag < len(values)
    average = statistics.mean(values)
    x = [v-average for v in values]
    denominator = sum(v*v for v in x)
    if denominator == 0:
        return [None]*(max_lag+1)
    return [sum(x[i]*x[i-lag] for i in range(lag,len(x)))/denominator for lag in range(max_lag+1)]


def portmanteau(values, lags=20):
    rho = acf(values,lags)
    if rho[0] is None:
        return None, None
    n = len(values)
    return n*sum(r*r for r in rho[1:]), n*(n+2)*sum(rho[k]**2/(n-k) for k in range(1,lags+1))


def shuffle(values, seed=731):
    out = list(values)
    random = uniform(seed)
    for i in range(len(out)-1,0,-1):
        j = math.floor(next(random)*(i+1))
        out[i],out[j] = out[j],out[i]
    return out


def clustered_returns(seed=2524):
    random = normal_generator(seed)
    return [next(random)*(.005 if (i//50)%2 == 0 else .025) for i in range(600)]


def variance_mixture(p=.2, ratio=5):
    assert 0 <= p <= 1 and ratio >= 1
    variance = 1-p+p*ratio**2
    sd = math.sqrt(variance)
    low, high = 1/sd, ratio/sd
    normal = statistics.NormalDist()
    return {'variance': variance, 'sd': sd, 'low': low, 'high': high,
            'kurtosis': 3*((1-p)+p*ratio**4)/variance**2,
            'density': lambda x: (1-p)*normal.pdf(x/low)/low+p*normal.pdf(x/high)/high,
            'tail': lambda threshold: (1-p)*math.erfc(abs(threshold)/low/math.sqrt(2))+p*math.erfc(abs(threshold)/high/math.sqrt(2))}


def realized_variance(log_prices, stride=1):
    assert isinstance(stride,int) and stride > 0 and (len(log_prices)-1)%stride == 0
    returns = [log_prices[i]-log_prices[i-stride] for i in range(stride,len(log_prices),stride)]
    variance = sum(r*r for r in returns)
    return {'variance':variance, 'volatility':math.sqrt(variance), 'returns':returns, 'count':len(returns)}


def intraday_sample(noise_bps=3, seed=81):
    assert 0 <= noise_bps <= 10
    random, signs = normal_generator(seed), uniform(seed+1000)
    latent = [0]
    for _ in range(390):
        latent.append(latent[-1]+.01/math.sqrt(390)*next(random))
    eta = noise_bps/10000
    observed = [p+eta*(-1 if next(signs)<.5 else 1) for p in latent]
    return {'latent':latent, 'observed':observed, 'eta':eta, 'integrated_variance':.01**2}


def intraday_profile(news=True):
    raw = [1+3*math.exp(-i/6)+2*math.exp(-(77-i)/7)+(5*math.exp(-.5*((i-30)/1.2)**2) if news else 0) for i in range(78)]
    total = sum(raw)
    return [x/total for x in raw]

"""Independent stdlib calculations for the prices / stochastic process lessons."""
import math
import statistics


PRICE_A=[100,102,101,103,102,104]
PRICE_B=[100,98,103,99,106,104]

def price_returns(prices):
    assert len(prices)>1 and all(p>0 and math.isfinite(p) for p in prices)
    return [{'simple':p/prices[i]-1,'log':math.log(p/prices[i])} for i,p in enumerate(prices[1:])]

def summary_stats(values):
    stats=moments(values)
    m3=statistics.mean((x-stats['mean'])**3 for x in values)
    return dict(stats,skewness=m3/stats['variance']**1.5 if stats['variance']>0 else None)

def arma11(phi,theta,innovation_variance=1,max_lag=20):
    assert abs(phi)<1 and innovation_variance>0
    numerator=1+theta*theta+2*phi*theta
    first=(phi+theta)*(1+phi*theta)/numerator
    return {'variance':innovation_variance*numerator/(1-phi*phi),'acf':[1]+[first*phi**(k-1) for k in range(1,max_lag+1)]}

def simulate_arma(phi,theta,count=600,seed=303):
    arma11(phi,theta)
    random=normal_generator(seed);out=[];previous=previous_shock=0
    for i in range(count+1000):
        shock=next(random);value=phi*previous+shock+theta*previous_shock
        if i>=1000:out.append(value)
        previous,previous_shock=value,shock
    return out

def fractional_weights(d,count=20):
    weights=[1]
    for j in range(1,count):weights.append(weights[-1]*(j-1-d)/j)
    return weights

def arfima_acf(d,max_lag=20):
    assert -.5<d<.5
    rho=[1]
    for k in range(1,max_lag+1):rho.append(rho[-1]*(k-1+d)/(k-d))
    return rho

def calendar_acf(strength=1,noise_sd=.01,max_lag=15):
    means=[v*strength for v in [-.004,.001,.001,.001,.001]]
    average=statistics.mean(means);a=[v-average for v in means]
    between=statistics.mean(x*x for x in a);variance=between+noise_sd**2
    return {'means':means,'between':between,'variance':variance,'acf':[1]+[statistics.mean(a[d]*a[(d-k)%5] for d in range(5))/variance for k in range(1,max_lag+1)]}

def squared_linear_correlation(psi,c4=0,innovation_variance=1,lag=1):
    gamma0=innovation_variance*sum(x*x for x in psi)
    gamma=innovation_variance*sum(psi[j]*psi[j+lag] for j in range(max(0,len(psi)-lag)))
    variance=2*gamma0**2+c4*sum(x**4 for x in psi)
    assert variance>0
    return (2*gamma**2+c4*sum(psi[j]**2*psi[j+lag]**2 for j in range(max(0,len(psi)-lag))))/variance

def standardized_t_pdf(x,nu=5):
    assert nu>2
    coefficient=math.exp(math.lgamma((nu+1)/2)-math.lgamma(nu/2))/math.sqrt(math.pi*(nu-2))
    return coefficient*(1+x*x/(nu-2))**(-(nu+1)/2)


def close(a,b,tol=1e-10):
    assert math.isclose(a,b,rel_tol=tol,abs_tol=tol),(a,b)
print("Self-contained standard-library examples. All numerical series are hypothetical.")

Self-contained standard-library examples. All numerical series are hypothetical.


## ราคาที่เห็นกับผลตอบแทนที่ได้รับ

ราคา 100 บาทบอกมูลค่าต่อหุ้น ณ เวลาหนึ่ง ผลตอบแทนต้องเทียบกับราคาที่ซื้อและเงินที่ได้รับระหว่างถือ ถ้าซื้อที่ 100 ขายที่ 99 และรับปันผล 2 ผู้ลงทุนได้ผลตอบแทน 1% แม้ราคาหุ้นลดลง 1%

ก่อนวิเคราะห์ข้อมูล เราจึงต้องระบุว่าใช้ราคาอะไร วัดช่วงใด และรวมกระแสเงินสดใดบ้าง ข้อตกลงเหล่านี้มีผลต่อทั้งค่าเฉลี่ย ความผันผวน และการเปรียบเทียบสินทรัพย์

หน้านี้ครอบคลุมหัวข้อ Prices and Returns ในสารบัญบท 2 ของ Taylor โดยใช้ตัวอย่างที่คำนวณขึ้นใหม่ อ่านต่อที่ [Stochastic Processes](../stochastic-processes.html) เพื่อแยกข้อมูลที่เกิดขึ้นแล้วออกจากแบบจำลองที่ใช้สร้างและพยากรณ์ข้อมูล

## ดูราคาสองเส้นให้เทียบกันได้

ให้ \(P_t\) เป็นราคาปิดเมื่อสิ้นช่วง t และเรียงข้อมูลตามเวลา เราเรียกชุด \(P_0,P_1,\ldots,P_T\) ว่าอนุกรมเวลา หรือ time series ดัชนี t ในตัวอย่างนี้นับวันซื้อขาย วันที่ 1 กับวันที่ 2 จึงอาจมีวันหยุดคั่นอยู่

ใช้ราคาสมมติสองชุดซึ่งเริ่มและจบเท่ากัน เพื่อดูว่าระหว่างทางต่างกันอย่างไร

| วันซื้อขาย | ราคา A | ราคา B |
|---|---:|---:|
| 0 | 100 | 100 |
| 1 | 102 | 98 |
| 2 | 101 | 103 |
| 3 | 103 | 99 |
| 4 | 102 | 106 |
| 5 | 104 | 104 |

ทั้งสองชุดเพิ่มขึ้น 4% ในห้าวัน แต่ B ขึ้นลงแรงกว่า A หลายวัน ผลตอบแทนต้น–ปลายช่วงจึงยังไม่บอกความเสี่ยงระหว่างถือ

หากสินทรัพย์เริ่มที่ราคาต่างกัน ใช้ดัชนีฐาน 100 เพื่อเทียบการเปลี่ยนแปลงเชิงสัดส่วน

$$
I_t=100\frac{P_t}{P_0}.
$$

การตั้งฐานใหม่ไม่เปลี่ยนผลตอบแทน และไม่ใช่การปรับปันผลหรือแตกหุ้น ส่วนกราฟราคาแบบ logarithmic ใช้ระยะห่างเท่ากันแทนการเปลี่ยนแปลงในอัตราส่วนเท่ากัน เช่น 100 → 110 และ 200 → 220 ต่างเพิ่ม 10%



ราคาสมมติหกจุดให้ผลตอบแทนห้าค่า ใช้สาธิตวิธีอ่านกราฟ ไม่ใช้ประมาณพฤติกรรมตลาดจากตัวอย่างขนาดเล็กนี้

In [2]:
for name,prices in [('A',PRICE_A),('B',PRICE_B)]:
    values=price_returns(prices)
    simple=math.prod(1+r['simple'] for r in values)-1
    log_total=sum(r['log'] for r in values)
    close(simple,.04); close(math.expm1(log_total),.04)
    print(name,prices)
    print('Simple returns (%)',[round(r['simple']*100,6) for r in values])
    print(f'Total simple {simple:.6%}; total log {log_total:.6%}')
scaled=price_returns([p*.5 for p in PRICE_B])
for a,b in zip(price_returns(PRICE_B),scaled):close(a['log'],b['log'])

A [100, 102, 101, 103, 102, 104]
Simple returns (%) [2.0, -0.980392, 1.980198, -0.970874, 1.960784]
Total simple 4.000000%; total log 3.922071%
B [100, 98, 103, 99, 106, 104]
Simple returns (%) [-2.0, 5.102041, -3.883495, 7.070707, -1.886792]
Total simple 4.000000%; total log 3.922071%


## ตรวจข้อมูลก่อนคำนวณ

คำว่า closing price อาจหมายถึงราคาซื้อขายสุดท้าย ราคาประมูลปิด หรือราคาที่ผู้ให้บริการประเมินไว้ ต้องอ่านนิยามของแหล่งข้อมูลก่อนนำหลายชุดมาต่อกัน

| เรื่องที่ต้องตรวจ | ผลที่อาจเกิดกับอนุกรม | วิธีจัดการ |
|---|---|---|
| เวลา เขตเวลา และปฏิทิน | ราคาปิดคนละตลาดอาจบันทึกคนละเวลา แม้อยู่วันเดียวกัน | เก็บ timestamp พร้อมเขตเวลา และกำหนดช่วงถือให้ตรงกัน |
| วันหยุดกับข้อมูลหาย | การเติมราคาซ้ำสร้างผลตอบแทนศูนย์ | แยกวันไม่เปิดตลาดออกจากวันที่ขาดข้อมูล บันทึกวิธีเติมถ้าจำเป็น |
| ราคาค้างหรือซื้อขายบาง | ผลตอบแทนศูนย์หลายวันแล้วกระโดดเมื่อมี trade | ตรวจเวลาซื้อขายล่าสุด ปริมาณ และราคา bid–ask ประกอบ |
| ปันผลและแตกหุ้น | ราคาเปลี่ยนโดยไม่ได้สะท้อนกำไรขาดทุนของผู้ถือทั้งหมด | ตรวจ corporate actions และนิยาม adjusted price |
| รายชื่อสินทรัพย์ | ใช้เฉพาะบริษัทที่ยังอยู่วันนี้แล้วมองข้ามบริษัทที่ถูกเพิกถอน | ใช้รายชื่อที่ทราบ ณ เวลานั้นและรวม delisting returns หากมี |
| ข้อมูลที่ถูกแก้ย้อนหลัง | การทดสอบอาจใช้ข้อมูลที่ผู้ลงทุนยังไม่รู้ในวันตัดสินใจ | เก็บวันที่เผยแพร่และวันที่แก้ไข ใช้ข้อมูลแบบ point-in-time เมื่อจำเป็น |
| หน่วยและสกุลเงิน | บาทกับสตางค์ หรือ USD/JPY กับ JPY/USD ให้ตัวเลขต่างกัน | ระบุหน่วยและทิศทางการเสนอราคาไว้กับชุดข้อมูล |

ตัวอย่างแตกหุ้น 2 ต่อ 1: ก่อนแตกถือหนึ่งหุ้นราคา 100 หลังแตกถือสองหุ้นราคาหุ้นละ 50 มูลค่ายังเป็น 100 การใช้ราคาดิบ 100 → 50 จะรายงาน −50% อย่างผิดความหมาย ถ้าปรับราคาเดิมเป็น 50 แล้ว ผลตอบแทนของเหตุการณ์นี้เป็นศูนย์

Adjusted close ของแต่ละผู้ให้บริการอาจใช้วิธีปรับต่างกัน บางชุดรวมปันผลและสมมติการนำกลับไปลงทุนแล้ว การบวกปันผลเข้าไปอีกครั้งจะนับซ้ำ เก็บราคาดิบ เหตุการณ์ปรับราคา และวิธีคำนวณแยกกันเมื่อทำได้

ราคาศูนย์หรือติดลบใช้ log return ไม่ได้ อย่าลบรายการเหล่านี้โดยอัตโนมัติ ต้องตรวจว่าข้อมูลผิดหรือเป็นราคาที่เกิดขึ้นได้ในตลาดนั้น เช่น futures บางประเภท แล้วเลือกตัวแปรที่สอดคล้องกับสัญญา

## แปลงราคาสองชุดเป็นผลตอบแทน

เมื่อไม่มีปันผล simple return ของแต่ละวันคือ \(R_t=P_t/P_{t-1}-1\) ตัวอย่าง B วันที่ 2 เพิ่มจาก 98 เป็น 103 จึงได้ \(103/98-1\approx5.1020\%\) ต้องหารด้วยราคาวันก่อน ไม่ใช่ราคาเริ่มต้น 100 ทุกวัน

| วัน | Simple return A | Simple return B |
|---|---:|---:|
| 1 | 2.0000% | −2.0000% |
| 2 | −0.9804% | 5.1020% |
| 3 | 1.9802% | −3.8835% |
| 4 | −0.9709% | 7.0707% |
| 5 | 1.9608% | −1.8868% |

กราฟราคาเน้นมูลค่าสะสม ส่วนกราฟผลตอบแทนช่วยเห็นขนาดการเปลี่ยนแปลงแต่ละช่วง การเพิ่มหรือลดระดับราคาทั้งชุดด้วยตัวคูณเดียวกันเปลี่ยนกราฟราคา แต่คงผลตอบแทนเดิม ลองปรับราคาเริ่มต้นของ B ในตัวทดลองแล้วสลับไปดูผลตอบแทน

ราคาที่มีแนวโน้มไม่ได้รับรองว่าผลตอบแทนจะพยากรณ์ได้ และการแปลงเป็นผลตอบแทนก็ไม่ได้รับรอง stationarity ต้องตรวจพฤติกรรมของชุดที่แปลงแล้วอีกครั้ง

## Simple return, log return และผลตอบแทนสะสม

ให้ \(D_t\) เป็นปันผลต่อหุ้นที่รับเมื่อสิ้นช่วง โดยยังไม่หักต้นทุนซื้อขาย

$$
R_t=\frac{P_t+D_t-P_{t-1}}{P_{t-1}},\qquad
r_t=\log(1+R_t).
$$

Simple return เป็นสัดส่วนกำไรต่อเงินต้น ส่วน log return คือ logarithm ของอัตราส่วนความมั่งคั่งปลายช่วงต่อต้นช่วง ใช้ได้เมื่อ \(1+R_t>0\) ที่ R=−100% จะไม่มี log return ที่เป็นจำนวนจำกัด

สำหรับผลตอบแทนขนาดเล็ก ใช้การขยาย Taylor ได้ว่า

$$
\log(1+R)=R-\frac{R^2}{2}+\frac{R^3}{3}-\cdots.
$$

จึงประมาณ r≈R ได้ใกล้ศูนย์ แต่ที่ R=20% ค่า r≈18.2322% ความต่างไม่เล็กพอจะละเสมอไป ต้องเก็บค่าทศนิยม .20 ในการคำนวณ แล้วค่อยคูณ 100 เมื่อต้องการแสดงเปอร์เซ็นต์

ผลตอบแทนหลายช่วงคำนวณดังนี้ โดยถ้ามีปันผลให้สมมติการนำกลับไปลงทุนอย่างสอดคล้องกัน

$$
1+R_{1:T}=\prod_{t=1}^T(1+R_t),\qquad
r_{1:T}=\sum_{t=1}^Tr_t=\log(1+R_{1:T}).
$$

ขึ้น 20% แล้วลง 20% เหลือเงิน \(100(1.2)(0.8)=96\) ผลตอบแทนสะสมคือ −4% ค่าเฉลี่ยเลขคณิตของ simple returns เท่ากับศูนย์ แต่ค่าเฉลี่ยเรขาคณิตต่อช่วงคือ \(\sqrt{0.96}-1\approx-2.0204\%\)

สำหรับน้ำหนักพอร์ต \(w_i\) ณ ต้นช่วง ซึ่งรวมเป็น 1 ผลตอบแทน simple ของพอร์ตเท่ากับ \(R_p=\sum_iw_iR_i\) แล้วจึงหา \(r_p=\log(1+R_p)\) ผลรวมถ่วงน้ำหนักของ log returns โดยทั่วไปไม่เท่ากับ log return ของพอร์ต

อัตราผลตอบแทนต่อปีต้องระบุวิธีแปลง ถ้ามี T วันซื้อขายและใช้ 252 วันต่อปี อัตราเติบโตทบต้นที่เทียบเป็นรายปีคือ

$$
G_{\rm annual}=(1+R_{1:T})^{252/T}-1.
$$

นี่เป็นการเทียบอัตราจากช่วงที่สังเกต ไม่ใช่การพยากรณ์ปีถัดไป ส่วนการใช้ \(252\bar R\) เป็นค่าเฉลี่ยแบบเลขคณิตที่แปลงหน่วย มีความหมายต่างกัน

In [3]:
simple,log_return=return_pair(100,99,2)
close(simple,.01)
print(f'Dividend-inclusive: simple={simple:.6%}, log={log_return:.6%}')
print('Split 2:1: value before=',1*100,'value after=',2*50)
close(2*50/100-1,0)
compound=1.2*.8-1
geometric=math.sqrt(.96)-1
print(f'+20%, -20%: cumulative={compound:.6%}; geometric per period={geometric:.6%}')
close(compound,-.04)

Dividend-inclusive: simple=1.000000%, log=0.995033%
Split 2:1: value before= 100 value after= 100
+20%, -20%: cumulative=-4.000000%; geometric per period=-2.020410%


## เมื่อเปลี่ยนสินทรัพย์ นิยามผลตอบแทนต้องตามไปด้วย

ดัชนีราคาและดัชนี total return อาจเริ่มที่ฐานเดียวกันแต่แยกจากกันเมื่อมีปันผล เลือกชุดให้ตรงกับผลตอบแทนที่ผู้ลงทุนได้รับ ส่วนการเปรียบเทียบข้ามประเทศต้องระบุว่าจะวัดในสกุลเงินท้องถิ่นหรือสกุลเงินของผู้ลงทุน

หากสินทรัพย์ให้ผลตอบแทน \(R_{\rm local}\) และสกุลเงินท้องถิ่นแข็งค่าต่อสกุลเงินผู้ลงทุนในอัตรา \(R_{\rm FX}\) จะได้

$$
1+R_{\rm home}=(1+R_{\rm local})(1+R_{\rm FX}).
$$

ตัวอย่างสินทรัพย์เพิ่ม 5% และสกุลเงินเพิ่ม 2% ให้ผลตอบแทนในสกุลผู้ลงทุน \(1.05(1.02)-1=7.1\%\) หากกลับทิศทางการ quote FX log return จะเปลี่ยนเครื่องหมาย ส่วน simple return ต้องคำนวณ \(1/(1+R)-1\)

ตราสารหนี้มี coupon, accrued interest และการเปลี่ยนราคา การใช้ clean price อย่างเดียวอาจตกหล่นกระแสเงินสด ส่วน futures ต้องระบุการวางเงิน การ mark-to-market และวิธีต่อสัญญา การกระโดดที่วัน roll ใน continuous series อาจเกิดจากวิธีต่อข้อมูล จึงไม่เท่ากับผลตอบแทนที่ซื้อขายได้โดยตรง

สำหรับข้อมูลระหว่างวัน การใช้ trade price กับ midpoint ให้ผลต่างกันจาก bid–ask spread อ่านต่อใน [ข้อมูลความถี่สูง](../asset-returns-stylized-facts.html#high-frequency) ส่วนการเปรียบเทียบรายวัน รายสัปดาห์ และรายเดือนต้องใช้ช่วงข้อมูลเดียวกันก่อนแยกผลของความถี่

In [4]:
for fx in [.02,-.02]:
    home=(1.05)*(1+fx)-1
    print(f'Local +5%, FX {fx:+.1%}: home return {home:.3%}')
close(1.05*1.02-1,.071);close(1.05*.98-1,.029)
forward=.02;reverse=1/(1+forward)-1
close(math.log1p(reverse),-math.log1p(forward))
print(f'Reverse FX quote: simple {reverse:.6%}; log sign reverses exactly')

Local +5%, FX +2.0%: home return 7.100%
Local +5%, FX -2.0%: home return 2.900%
Reverse FX quote: simple -1.960784%; log sign reverses exactly


## ลองคำนวณ

1. ใช้ราคาทั้งสองชุดในตาราง คำนวณผลตอบแทนสะสมจากผลคูณ simple returns และจากผลบวก log returns ผลตรงกันหรือไม่?
2. ซื้อหนึ่งหุ้นที่ 100 รับปันผล 2 และขายที่ 99 ได้ผลตอบแทนเท่าไร? ถ้าผู้ให้ข้อมูลปรับปันผลไว้แล้ว จะป้องกันการนับซ้ำอย่างไร?
3. ถือสินทรัพย์ที่ให้ผลตอบแทน 5% แต่สกุลเงินท้องถิ่นอ่อนลง 2% ผลตอบแทนในสกุลผู้ลงทุนเป็นเท่าไร?

**เปิดแนวคำตอบ**

ข้อ 1 ทั้งสองชุดได้ simple return สะสม 4% และ log return สะสม \(\log(1.04)\approx3.9221\%\)

ข้อ 2 ได้ simple return 1% และ log return ประมาณ 0.9950% ต้องอ่านนิยาม adjusted price และเลือกคำนวณจากราคาที่รวมปันผลแล้ว หรือจากราคาดิบกับกระแสเงินสด โดยไม่รวมสองวิธีซ้ำกัน

ข้อ 3 ได้ \(1.05(0.98)-1=2.9\%\)

[ดาวน์โหลด Notebook](prices-and-returns.ipynb) มีราคาสองชุด ตารางผลตอบแทน ตัวอย่างปันผล แตกหุ้น การทบต้น และการแปลงสกุลเงิน พร้อมผลรัน

## แหล่งอ้างอิงและบทถัดไป

- Stephen J. Taylor, *Asset Price Dynamics, Volatility, and Prediction* (2005), หัวข้อบท 2 · [สารบัญและคำนำจากเว็บไซต์ผู้เขียน](https://www.lancaster.ac.uk/people/afasjt/apdvp_contents.pdf)
- [หน้าหนังสือและทรัพยากรของผู้เขียน](https://www.lancaster.ac.uk/staff/afasjt/assetpricedynamics.html)
- [Stochastic Processes](../stochastic-processes.html) สำหรับตัวแปรสุ่ม stationarity และแบบจำลองอนุกรมเวลา
- [ตารางเทียบหัวข้อบท 2–4](../asset-returns-stylized-facts.html#coverage-map)

ตัวอย่างตัวเลขและภาพในหน้านี้สร้างใหม่เพื่อการเรียนรู้ ไม่ใช่การคัดลอกอนุกรมราคาหรือตารางสถิติจากหนังสือ